# FeynmanEngine — Getting Started

5-minute walk-through:

1. Install
2. Verify what's set up (`feynman doctor`)
3. Generate your first Feynman diagram
4. Compute the spin-averaged $|\overline{\mathcal{M}}|^2$ and integrated $\sigma$
5. Launch the browser UI

Once you're comfortable, jump to one of the audience-targeted notebooks:
- **`for_undergrad_qft.ipynb`** — diagram + amplitude examples for teaching particle physics.
- **`for_lhc_experimentalist.ipynb`** — LHC observables, K-factors, comparison to data.
- **`for_bsm_theorist.ipynb`** — register a custom |M̄|² for your own BSM operator.
- **`nlo_quickstart.ipynb`** — five-step tour of the NLO machinery.

## 1. Install

FeynmanEngine is on PyPI. The Python package is small (~9 MB); the native HEP tools (QGRAF, FORM, LoopTools, LHAPDF, OpenLoops 2) are compiled from bundled source archives via `feynman setup` after install.

**Three install paths — pick one:**

### Recommended (full functionality)
```bash
pip install feynman-engine
feynman setup        # ~10–15 min, one-time. Builds QGRAF + FORM + LoopTools + LHAPDF + OpenLoops 2
```

### Minimal (teaching, CI without `gfortran`, faster install)
```bash
pip install feynman-engine
feynman setup --skip-lhapdf --skip-openloops    # ~3 min. QGRAF + FORM + LoopTools only
```
Hadronic σ uses the built-in LO PDF (factor-of-2-3 accuracy); NLO σ for unregistered QCD processes returns `HTTP 422` with a hint to install OpenLoops.

### Docker (everything bundled — no build needed)
```bash
docker run -p 8000:8000 ecavan/feynman-api:latest
# Open http://localhost:8000
```

**For SVG diagram rendering** (any path) you also need `lualatex` + `pdf2svg`:
- macOS: `brew install basictex pdf2svg && tlmgr install tikz-feynman`
- Debian/Ubuntu: `apt-get install texlive-luatex texlive-pictures texlive-science pdf2svg`

## 2. Verify what's installed

`feynman doctor` reports each component and recommends the next install command if anything is missing.

In [ ]:
from feynman_engine.diagnostics import collect_diagnostics, summarize_doctor_report

print(summarize_doctor_report(collect_diagnostics()))

Each component independently reports `ok` or `missing`. Missing native deps don't break the package — the engine falls back gracefully or returns a `HTTP 422` with a workaround.

**Don't have `feynman doctor` on the command line yet?** That's expected if you ran the cells above before pip-installing locally. Just run `pip install feynman-engine` in your terminal first, then come back.

## 3. Your first Feynman diagram

The classic textbook process: $e^+ e^- \to \mu^+ \mu^-$ at tree level (one $s$-channel photon).

In [ ]:
from feynman_engine.engine import FeynmanEngine

engine = FeynmanEngine()
result = engine.generate(
    process="e+ e- -> mu+ mu-",
    theory="QED",
    loops=0,
    output_format="svg",
)

print(f"Generated {result.summary['total_diagrams']} diagram(s):")
for d in result.diagrams:
    print(f"  Diagram {d.id}: topology = {d.topology}")

In [ ]:
# Render the first diagram inline (requires lualatex + pdf2svg installed)
from IPython.display import SVG, display

first_id = result.diagrams[0].id
svg_bytes = result.images.get(first_id)
if svg_bytes:
    display(SVG(svg_bytes))
else:
    print("SVG rendering unavailable — install lualatex + pdf2svg (see Install section).")

## 4. Symbolic $|\overline{\mathcal{M}}|^2$ and integrated $\sigma$

The engine returns the spin-averaged matrix-element-squared as a SymPy expression, then integrates it to a cross-section in pb.

## 5. Launch the browser UI

The interactive UI is the friendliest way to explore the engine. It bundles:

- **120+ pre-loaded example processes** organised by theory in the left sidebar (QED, QCD, QCD+QED, EW, BSM)
- **Diagram rendering** with TikZ-Feynman → SVG
- **|M̄|² and integral display** with KaTeX math typesetting
- **Cross-section integration** at any √s, LO and NLO
- **Differential histograms** for cosθ, pT, η, y, M_inv, M_ll, ΔR
- **Trust-level badges** (validated/approximate/blocked) on every result
- **Swagger docs** at `/docs` for the full REST API

**From the command line:**
```bash
feynman serve
# Then open http://localhost:8000
```

**Or launch from this notebook** (cell below). Sometimes embedded iframes are blocked by browser security; if the inline view is empty, click the **Open in browser** link.

## 5. Launch the browser UI

The interactive UI bundles diagram rendering, $|\overline{\mathcal{M}}|^2$ display with KaTeX, cross-section integration, differential histograms, and 120+ pre-loaded example processes organised by theory.

From the command line:
```bash
feynman serve
# Open http://localhost:8000
```

Or launch from the notebook (cell below). Sometimes embedded iframes are blocked by browser security; if the inline view is empty, click the **Open in browser** link.

In [ ]:
import subprocess, sys, time
from IPython.display import IFrame, Markdown, display

server = subprocess.Popen(
    [sys.executable, "-m", "feynman_engine", "serve",
     "--host", "127.0.0.1", "--port", "8765"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(3)

display(Markdown("### → [Open in browser](http://localhost:8765)"))
display(IFrame("http://localhost:8765", width="100%", height=600))

In [ ]:
# Stop the server when done.
server.terminate()
server.wait()
print("Server stopped.")

## What's next

- **`for_undergrad_qft.ipynb`** — diagram + amplitude examples for teaching particle physics.
- **`for_lhc_experimentalist.ipynb`** — LHC observables, K-factors, comparison to ATLAS/CMS data.
- **`for_bsm_theorist.ipynb`** — register a custom |M̄|² for your own BSM operator.
- **`nlo_quickstart.ipynb`** — five-step tour of the NLO machinery (universal QED, EW Sudakov, hadronic DY).
- **Browser UI**: 120+ curated examples in the sidebar, organised by theory (QED, QCD, EW, BSM).
- **Swagger docs**: http://localhost:8000/docs.
- **Citations + bundled tools**: see `README.md`.